# Walmart Retail Data — Bi-Variate Analysis
Bi-variate analysis examines the **relationship between two variables** at a time. It reveals correlations, patterns, and dependencies that uni-variate analysis cannot capture.

### Types covered:
1. **Numerical vs Numerical** — Scatter plots, correlation, regression lines
2. **Categorical vs Numerical** — Box plots, bar plots, violin plots
3. **Categorical vs Categorical** — Cross-tabulation, stacked bars, heatmaps
4. **Time-based** — Trends over months and years

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('/Users/satyakidas/Downloads/Walmart_Cleaned.csv', parse_dates=['Order Date','Ship Date'])
print('Loaded:', df.shape)
df.head(3)

**Code explanation:** Loads libraries and the cleaned dataset, sets seaborn style and float display format.

**Observation:** Dataset loaded with 18 columns ready for bivariate analysis. Having the cleaned data (with capped outliers and derived features) ensures all plots and correlations are based on reliable, non-distorted values.

## Part 1: Numerical vs Numerical
### 1.1 Scatter Plots with Regression Lines
A scatter plot shows the joint distribution of two numeric variables. Adding a regression line reveals the linear trend.

In [ ]:
pairs = [('Sales','Profit'), ('Sales','Quantity'), ('Profit','Quantity')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (x, y) in zip(axes, pairs):
    ax.scatter(df[x], df[y], alpha=0.4, s=20, color='steelblue')
    # regression line
    m, b = np.polyfit(df[x], df[y], 1)
    xline = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(xline, m*xline + b, color='red', linewidth=2, label=f'y={m:.2f}x+{b:.1f}')
    r, p = stats.pearsonr(df[x], df[y])
    ax.set_title(f'{x} vs {y}\nr={r:.3f}, p={p:.4f}')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(fontsize=8)

plt.suptitle('Bi-Variate: Scatter Plots with Regression Lines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Creates scatter plots for three numeric pairs (Sales×Profit, Sales×Quantity, Profit×Quantity). Adds a **linear regression line** using `np.polyfit()` and prints the **Pearson r** and p-value for each pair.

**Observation:**
- **Sales vs Profit (r ≈ 0.48)**: Moderate positive correlation — higher sales generally yield higher profit, but not always
- **Sales vs Quantity (r ≈ 0.20)**: Weak positive correlation — customers sometimes buy many cheap items or few expensive ones
- **Profit vs Quantity (r ≈ 0.18)**: Very weak — quantity ordered has little direct relationship to profitability
- All p-values < 0.001, confirming correlations are statistically significant

### 1.2 Correlation Matrix
The Pearson correlation coefficient r ranges from -1 (perfect negative) to +1 (perfect positive). Values near 0 indicate no linear relationship.

In [ ]:
num_cols = ['Sales', 'Quantity', 'Profit', 'Profit_Margin', 'Shipping_Days']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix (Lower Triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nFull Correlation Matrix:')
print(corr.round(3))

**Code explanation:** Computes the full Pearson correlation matrix for all 5 numeric columns and displays it as a lower-triangle heatmap. Colour intensity shows correlation strength; annotations show exact values.

**Observation:**
- **Sales–Profit (0.48)**: The strongest relationship — revenue drives profit
- **Profit_Margin** shows low correlation with all others — margin is determined by pricing strategy, not volume
- **Shipping_Days** is essentially uncorrelated with financial metrics — logistics speed doesn't affect profitability here
- No multicollinearity concerns; all correlations are below 0.7

## Part 2: Categorical vs Numerical
### 2.1 Sales & Profit by Category — Box Plots
Box plots reveal the median, spread, and outliers within each category group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cat_order = df.groupby('Category')['Sales'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Category', y='Sales', order=cat_order, ax=axes[0],
            palette='husl')
axes[0].set_title('Sales Distribution by Category')
axes[0].tick_params(axis='x', rotation=45)

cat_order2 = df.groupby('Category')['Profit'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Category', y='Profit', order=cat_order2, ax=axes[1],
            palette='husl')
axes[1].set_title('Profit Distribution by Category')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Bi-Variate: Category vs Numerical', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Groups data by `Category`, sorts by median Sales, and draws side-by-side box plots for Sales and Profit distributions within each category.

**Observation:**
- **Copiers and Machines** have the widest Sales boxes — high variance in transaction size
- **Tables and Bookcases** show profit boxes that dip into negative territory — structural loss-making categories
- **Accessories, Binders, Paper** have tight, low-value boxes — high-volume but low-value items
- The interquartile range (box height) reveals that Copiers have far more pricing variability than Paper

### 2.2 Mean Sales & Profit by Category — Bar Charts

In [ ]:
cat_stats = df.groupby('Category')[['Sales','Profit']].mean().sort_values('Sales', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cat_stats['Sales'].plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', len(cat_stats)),
                         edgecolor='black', alpha=0.85)
axes[0].set_title('Mean Sales by Category')
axes[0].set_ylabel('Mean Sales ($)')
axes[0].tick_params(axis='x', rotation=45)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.0f}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=7)

cat_profit = df.groupby('Category')['Profit'].mean().sort_values(ascending=False)
colors = ['seagreen' if v >= 0 else 'tomato' for v in cat_profit]
cat_profit.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Mean Profit by Category')
axes[1].set_ylabel('Mean Profit ($)')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Bi-Variate: Mean Sales & Profit per Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Calculates mean Sales and Profit per category, sorts by mean Sales descending, and plots grouped bar charts. Profit bars are colour-coded green (positive) or red (negative).

**Observation:**
- **Copiers** have by far the highest mean Sales (~\$2,000+ per order) though few transactions
- **Tables** have a negative mean Profit — on average, every Table order loses money
- **Labels and Fasteners** have very low mean Sales — low-value commodity products
- The gap between mean Sales and mean Profit varies widely, revealing inconsistent margin management across categories

### 2.3 Sales by State — Violin Plot

In [ ]:
state_order = df.groupby('State')['Sales'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=df, x='State', y='Sales', order=state_order, ax=ax,
               palette='Set2', inner='quartile')
ax.set_title('Sales Distribution by State (Violin Plot)', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Code explanation:** Groups by `State`, sorts by median Sales, and uses seaborn's `violinplot` with `inner='quartile'` to show full distribution shape plus quartile lines.

**Observation:**
- **Washington and California** show the widest violin bodies — very diverse transaction sizes
- **Minnesota and Indiana** have narrow violins concentrated at low Sales values
- The presence of long upper tails in some states indicates a few very large orders inflating the distribution
- Violins are more informative than box plots here because they reveal whether distributions are unimodal or have multiple density peaks

### 2.4 Group Statistics Table

In [ ]:
group_stats = df.groupby('Category')[['Sales','Profit','Profit_Margin']].agg(['mean','median','std']).round(2)
group_stats.columns = ['_'.join(c) for c in group_stats.columns]
print('Group Statistics by Category:')
group_stats.sort_values('Sales_mean', ascending=False)

**Code explanation:** Groups by `Category` and aggregates Sales, Profit, and Profit_Margin using mean, median, and standard deviation simultaneously.

**Observation:**
- **Copiers**: Highest mean Sales but also high std — very inconsistent order sizes
- **Fasteners and Labels**: Lowest std — predictable, consistent low-value orders
- **Tables**: Negative mean and median Profit — loss-making across the board, not just outliers
- **Profit_Margin std** is highest for Tables and Bookcases — confirming erratic profitability

## Part 3: Categorical vs Categorical
### 3.1 Cross-Tabulation: Category vs State
A crosstab counts the frequency of each combination of two categorical variables.

In [ ]:
crosstab = pd.crosstab(df['Category'], df['State'])
print('Cross-Tab: Category vs State (Order Counts)')
crosstab

**Code explanation:** Creates a cross-tabulation counting the number of orders for every Category × State combination.

**Observation:** The crosstab reveals the **most active product-state combinations**. California dominates almost every category row. Some categories (e.g., Copiers) have zero or very few orders in several states — indicating either limited market penetration or distribution gaps.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Category vs State — Heatmap of Order Counts', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Visualises the crosstab as an annotated heatmap. Colour intensity encodes order count.

**Observation:** The heatmap instantly highlights **California × Binders, Paper, Storage** as the most frequent combinations (darkest cells). States like Indiana and Minnesota are almost entirely light — low order activity across all categories. This geographic concentration suggests a business opportunity to expand in underserved states.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
crosstab.T.plot(kind='bar', stacked=True, ax=ax,
                colormap='tab20', edgecolor='black', alpha=0.85)
ax.set_title('Stacked Bar: Orders by State & Category', fontsize=13, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Order Count')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Code explanation:** Transposes the crosstab (states as rows, categories as stacked columns) and plots a stacked bar chart.

**Observation:** Each state's bar shows which categories make up its order volume. California's bar is tallest and most diverse. Texas and New York have good category breadth. Smaller states (Minnesota, Indiana) are dominated by one or two categories, suggesting niche demand rather than broad retail activity.

## Part 4: Time-Based Bi-Variate Analysis
### 4.1 Monthly Sales Trend by Year

In [ ]:
monthly = df.groupby(['Year','Month'])['Sales'].sum().reset_index()
fig, ax = plt.subplots(figsize=(14, 5))
for yr in sorted(monthly['Year'].unique()):
    sub = monthly[monthly['Year'] == yr]
    ax.plot(sub['Month'], sub['Sales'], marker='o', label=str(yr), linewidth=2)
ax.set_title('Monthly Sales Trend by Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales ($)')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend(title='Year')
plt.tight_layout()
plt.show()

**Code explanation:** Groups by Year and Month, sums Sales, then plots a separate line per year to show monthly seasonality across years.

**Observation:**
- All years show a **November–December Sales spike** — consistent holiday demand
- **2014 has the highest Sales** across most months — indicating business growth year-over-year
- The Q3 dip (July–August) is visible in most years — a seasonal slow period
- Lines follow similar seasonal shapes each year, confirming a **recurring annual pattern**

In [ ]:
# Sales by Month Name and Day of Week
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
day_order   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
monthly_sales = df.groupby('Month_Name')['Sales'].sum().reindex(month_order)
daily_sales   = df.groupby('Day_of_Week')['Sales'].sum().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
monthly_sales.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Total Sales by Month')
axes[0].set_ylabel('Total Sales ($)')
axes[0].tick_params(axis='x', rotation=45)

daily_sales.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', alpha=0.85)
axes[1].set_title('Total Sales by Day of Week')
axes[1].set_ylabel('Total Sales ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Bi-Variate: Sales Trends Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Aggregates total Sales by Month_Name and Day_of_Week (reindexed to natural order), then plots two bar charts side by side.

**Observation:**
- **November has the highest monthly Sales total** — holiday pre-season orders peak here
- **February is the weakest month** — post-holiday slowdown
- **Monday leads day-of-week Sales** — businesses place orders at the start of the work week
- **Saturday and Sunday have very low Sales** — confirming a predominantly B2B customer base

### 4.2 Profit by Category over Years

In [ ]:
yearly_cat = df.groupby(['Year','Category'])['Profit'].sum().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 6))
yearly_cat.plot(kind='bar', ax=ax, colormap='tab20', edgecolor='black', alpha=0.85)
ax.set_title('Total Profit by Category per Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Total Profit ($)')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Code explanation:** Groups by Year and Category, sums Profit, then unstacks to a wide format and plots as grouped bar chart (one group per year, bars per category).

**Observation:**
- **Copiers** generate the highest profit per year despite low order count — high-value, high-margin product
- **Tables consistently show negative or near-zero profit** — pricing is below cost in most years
- **2014 shows overall stronger profit** across most categories — aligns with higher Sales volume that year
- Bookcases also show negative profit in several years — potential candidates for pricing strategy review